In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)


import gc
import ast
import torch
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset
from peft import PeftModel
from peft import LoraConfig
from peft import get_peft_model
from transformers import Trainer
from transformers import TrainingArguments
from transformers import Blip2Processor, Blip2ForConditionalGeneration

In [ ]:
train_img_root = './blip/data/train'
test_img_root = './blip/data/test'

# 1. 基本加载
df = pd.read_csv('./blip/data/processed_dataset.csv')
df['caption'] = df['raw'].apply(ast.literal_eval)  # 将字符串转换为列表
# 2. 根据 'split' 列进行划分
train_df = df[df['split'] == 'train']
# print(train_df.head())  # 查看前5行
test_df = df[df['split'] == 'test']
# print(test_df.head())  # 查看前5行

train_data = []
train_df = train_df[:16]  # 重置索引，确保连续
for i in range(len(train_df)):
    img_path = train_df.iloc[i]['filename']
    caption = train_df.iloc[i]['caption']
    for cap in caption:
        train_data.append({
            "image": os.path.join(train_img_root, img_path),
            "text": cap.lower()
        })  
print(train_data)
print("Length of train_data:", len(train_data))

test_data = []

for i in range(len(test_df)):
    img_path = test_df.iloc[i]['filename']
    caption = test_df.iloc[i]['caption']
    for cap in caption:
        test_data.append({
            "image": os.path.join(test_img_root, img_path),
            "text": cap.lower()
        })
print(test_data) 
print("Length of test_data:", len(test_data))

In [ ]:
model_name = "Salesforce/blip2-flan-t5-xl"
cache_dir = "./model/blip/blip2_cache"
dtype = torch.bfloat16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# processor
processor = Blip2Processor.from_pretrained(
    model_name,
    cache_dir=cache_dir
    )
# model
model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=dtype,
    ).to(device)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
print(type(model.language_model))
for name, module in model.named_modules():
    if "language_model" in name:
        print(name)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        # ===== Q-Former =====
        "query",
        "key",
        "value",

        # ===== T5 =====
        "q",
        "k",
        "v",
        "o",   # optional
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"模型设备检查: {next(model.parameters()).device}")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)


In [ ]:
class FlickrDataset(Dataset):
    def __init__(self, data, processor, max_length=128):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # load image
        image = Image.open(item["image"]).convert("RGB")
        caption = item["text"]
        
        prompt = "Question: Describe this photo. Answer:"

        # 🔥 正确方式：一起处理
        encoding = self.processor(
            images=image,
            text=prompt,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=32
        )

        # labels 只给 caption
        labels = self.processor.tokenizer(
            caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=32
        ).input_ids

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        encoding["labels"] = labels
        return {k: v.squeeze(0) for k, v in encoding.items()}

# ==========================================
# 验证修正后的效果
# ==========================================
# 实例化
train_dataset = FlickrDataset(train_data, processor)
sample = train_dataset[0]

print("Input IDs shape:", sample['input_ids'].shape)
print("Labels shape:", sample['labels'].shape)

# 检查 Labels 中有多少有效的 token (不是 -100)
valid_labels = sample['labels'][sample['labels'] != -100]
print(f"有效 Label 数量: {len(valid_labels)}")

# 解码查看
print("Decoded Input:", processor.decode(sample['input_ids'], skip_special_tokens=True))
print("Decoded Labels (Only Caption part):", processor.decode(valid_labels, skip_special_tokens=True))

training_args = TrainingArguments(
    output_dir="./blip/blip2_lora",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=100,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

In [ ]:
model.eval()  # 推理模式

def generate_caption(sample):
    pixel_values = sample['pixel_values'].unsqueeze(0).to(device)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values=pixel_values, max_new_tokens=32)
    caption = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return caption

# 测试训练集前5张图片
for i in range(50):
    sample = train_dataset[i]
    caption = generate_caption(sample)
    print(f"Image {i} -> Generated caption: {caption}")

In [ ]:
model.save_pretrained("./blip/blip2_lora")
processor.save_pretrained("./blip/blip2_lora")